# 02 — RQ1: pose estimation under the aerial domain gap
How much do COCO-pretrained pose models degrade from above, and how much does
fine-tuning recover?

- **Part A** — sanity reference: PCK of RTMPose on *ground-level* COCO val.
- **Part B** — aerial PCK on **UAV-Human** (needs free registration:
  https://sutdcv.github.io/uav-human-web/ — request the *pose estimation subset*).
- **Part C** — no-registration path: pseudo-label Okutama with a large teacher,
  fine-tune yolo11n-pose as student, evaluate on held-out video.


### Setup (every notebook starts with this)
1. Runtime → Change runtime type → **T4 GPU** (free tier).
2. Zip your local `Project/` folder's `src/` and `scripts/` dirs as `src.zip`
   (`cd Project && zip -r src.zip src scripts`), then either upload it below
   or put it in Drive and adjust `SRC_ZIP`.


In [ ]:
# --- environment ---
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch, os
print('cuda:', torch.cuda.is_available())

# --- project code: upload src.zip (or mount Drive and set SRC_ZIP) ---
from pathlib import Path
SRC_ZIP = None  # e.g. '/content/drive/MyDrive/sar_project/src.zip'
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()  # choose src.zip
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')
print('project code ready')

# --- results go to Drive so they survive the session ---
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Part A — ground-level reference PCK (evaluator sanity check)
import config
from pathlib import Path
config.RAW_DIR = Path('/content/data/raw'); config.RAW_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR = OUT
!curl -sL -o /content/coco_ann.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!cd /content/data/raw && unzip -q -o /content/coco_ann.zip annotations/person_keypoints_val2017.json
import eval_pose
eval_pose.RAW_DIR = config.RAW_DIR; eval_pose.TABLES_DIR = OUT
ref = eval_pose.validate_on_coco(max_persons=150, device='cuda')


In [ ]:
# Okutama-Action direct downloads (public Dropbox folder, verified working).
# preview=<file>&dl=1 selects a single file from the shared folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
# Part B — UAV-Human aerial PCK (public Google Drive, no registration needed)
!pip -q install gdown
!gdown 1kWStmFjrN1Njf6mj4rTso6XPMULcFKS5 -O /content/PoseEstimation.zip
!mkdir -p /content/data/raw/uavhuman_pose && unzip -q -o /content/PoseEstimation.zip -d /content/data/raw/uavhuman_pose


In [ ]:
# Zero-shot aerial PCK on ALL 22,476 frames (project loader; ~30 min on T4).
# Ablation built in: `downscale` shrinks people before pose to probe the
# scale axis of the domain gap (0.5 -> half-size people), not just viewpoint.
import numpy as np, cv2
from data.uavhuman import iter_dataset
from pose import PoseEstimator
from eval_pose import pck

def aerial_pck(limit=None, downscale=1.0, device='cuda'):
    est = PoseEstimator(device=device)
    preds, gts, viss, boxes = [], [], [], []
    for img_path, persons in iter_dataset('/content/data/raw/uavhuman_pose', limit=limit):
        img = cv2.imread(str(img_path))
        if img is None: continue
        if downscale != 1.0:
            img = cv2.resize(img, None, fx=downscale, fy=downscale)
        for p in persons:
            box = p['box'] * downscale
            kp, _ = est(img, box[None])
            preds.append(kp[0]); gts.append(p['kpts'] * downscale)
            viss.append(p['vis']); boxes.append(box)
    return pck(np.stack(preds), np.stack(gts), np.stack(viss), np.stack(boxes))

aerial = aerial_pck(limit=None)             # full zero-shot aerial PCK
aerial_small = aerial_pck(limit=3000, downscale=0.35)  # small-person regime
print('AERIAL zero-shot:', aerial)
print('AERIAL @0.35 scale:', aerial_small)  # compare both against Part A


In [ ]:
# Part C — pseudo-label Okutama -> fine-tune yolo11n-pose (student)
import numpy as np, cv2, sys
fetch_okutama('Sample.zip')          # quick run; use TrainSetVideos.zip for the real one
from data.okutama import parse_annotations
from pose import PoseEstimator

video = '/content/data/okutama/1.1.1.mov'
labels = parse_annotations('/content/data/okutama/1.1.1.txt')
est = PoseEstimator(device='cuda')   # teacher (swap in RTMPose-l for a stronger teacher)

# Build a YOLO-pose dataset: crops around GT boxes + teacher keypoints
from pathlib import Path
ds = Path('/content/data/okutama_pose'); (ds/'images/train').mkdir(parents=True, exist_ok=True)
(ds/'labels/train').mkdir(parents=True, exist_ok=True)
(ds/'images/val').mkdir(parents=True, exist_ok=True); (ds/'labels/val').mkdir(parents=True, exist_ok=True)
cap = cv2.VideoCapture(video); idx = 0; n = 0
while True:
    ok, img = cap.read()
    if not ok: break
    if idx in labels and idx % 5 == 0:
        H, W = img.shape[:2]
        boxes = np.array([b.box for b in labels[idx]], np.float32)
        kpts, scores = est(img, boxes)
        split = 'val' if idx % 25 == 0 else 'train'
        lines = []
        for (x1,y1,x2,y2), kp, sc in zip(boxes, kpts, scores):
            if sc.mean() < 0.35: continue   # keep confident pseudo-labels only
            cx,cy,w,h = ((x1+x2)/2/W,(y1+y2)/2/H,(x2-x1)/W,(y2-y1)/H)
            ks = ' '.join(f'{x/W:.5f} {y/H:.5f} {2 if s>0.35 else 0}' for (x,y),s in zip(kp,sc))
            lines.append(f'0 {cx:.5f} {cy:.5f} {w:.5f} {h:.5f} ' + ks)
        if lines:
            cv2.imwrite(str(ds/f'images/{split}/{idx:06d}.jpg'), img)
            (ds/f'labels/{split}/{idx:06d}.txt').write_text('\n'.join(lines))
            n += 1
    idx += 1
cap.release(); print(n, 'pseudo-labeled frames')
(ds/'okutama_pose.yaml').write_text(f'path: {ds}\ntrain: images/train\nval: images/val\nkpt_shape: [17, 3]\nnames:\n  0: person\n')


In [ ]:
# Student fine-tune + PCK before/after vs teacher on the val frames
from ultralytics import YOLO
student = YOLO('yolo11n-pose.pt')
student.train(data=str(ds/'okutama_pose.yaml'), epochs=20, imgsz=1280, batch=8,
              device=0, project='/content/runs', name='pose_ft', exist_ok=True)
!cp /content/runs/pose_ft/weights/best.pt {OUT}/yolo11n_pose_okutama.pt


In [ ]:
# Stratified PCK plot (the RQ1 headline figure)
import json, matplotlib.pyplot as plt
# fill with your measured values: ground-level (Part A), aerial zero-shot,
# aerial fine-tuned (Part B/C)
results = {'ground-level (COCO)': ref}
fig, ax = plt.subplots(figsize=(6,4))
bins = [k for k in ref if k.startswith('PCK_h')]
for name, m in results.items():
    ax.plot(bins, [m[b] for b in bins], marker='o', label=name)
ax.set_xlabel('person pixel height'); ax.set_ylabel(f"PCK@{ref['alpha']}")
ax.set_title('RQ1: pose accuracy vs person scale'); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT / 'rq1_pck_vs_scale.png', dpi=150)
